# Setup and the schema

In [5]:
import os, time
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types
from openai import OpenAI

load_dotenv(override=True)
GEMINI_MODEL = os.environ["GEMINI_MODEL"]
GROQ_MODEL = os.environ["GROQ_MODEL"]

gemini_client = genai.Client()
groq_client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url=os.environ["GROQ_BASE_URL"],
)

class Ticket(BaseModel):
    category: Literal["billing", "bug", "feature_request", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="One sentence, under 25 words")
    needs_human: bool
    order_id: str | None  # required key, but the value may be null

TEXT = (
    "Hi, I was charged twice for order #A-10492 this morning and my card "
    "is now overdrawn. Please fix this today, I need the money back."
)
PROMPT = f"Extract a support ticket from this message:\n\n{TEXT}"

# Gemini, with the Pydantic class passed straight in

In [6]:
t0 = time.perf_counter()
g_resp = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=PROMPT,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Ticket,
    ),
)
print(f"{time.perf_counter() - t0:.2f}s")
print(g_resp.text)         # raw JSON string
print(g_resp.parsed)       # already a Ticket instance
print(g_resp.usage_metadata)

1.02s
{
  "category": "billing",
  "urgency": "high",
  "summary": "The customer was charged twice for order #A-10492, causing their card to be overdrawn and requesting a refund.",
  "needs_human": true,
  "order_id": "A-10492"
}
category='billing' urgency='high' summary='The customer was charged twice for order #A-10492, causing their card to be overdrawn and requesting a refund.' needs_human=True order_id='A-10492'
cache_tokens_details=None cached_content_token_count=None candidates_token_count=79 candidates_tokens_details=None prompt_token_count=47 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=47
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=126 traffic_type=None


# Groq, which wants the JSON Schema spelled out
Groq’s strict mode requires every property to be listed under required and no extra properties allowed. Pydantic doesn’t emit that by default, so this small helper adds it:

In [7]:
def strict_schema(model: type[BaseModel]) -> dict:
    schema = model.model_json_schema()
    schema["additionalProperties"] = False
    schema["required"] = list(schema["properties"])
    return schema

t0 = time.perf_counter()
q_resp = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "ticket", "strict": True, "schema": strict_schema(Ticket)},
    },
)
print(f"{time.perf_counter() - t0:.2f}s")
raw = q_resp.choices[0].message.content
print(raw)
print(Ticket.model_validate_json(raw))   # you validate it yourself
print(q_resp.usage)

1.11s
{"category":"billing","urgency":"high","summary":"Charged twice for order #A-10492; need refund today.","needs_human":true,"order_id":"A-10492"}
category='billing' urgency='high' summary='Charged twice for order #A-10492; need refund today.' needs_human=True order_id='A-10492'
CompletionUsage(completion_tokens=240, prompt_tokens=307, total_tokens=547, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=187, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.087942262, prompt_time=0.045143308, completion_time=0.50536104, total_time=0.550504348)


# Failure example

In [8]:
try:
    Ticket.model_validate_json('{"category": "refund", "urgency": "high"}')
except ValidationError as e:
    print(e)

4 validation errors for Ticket
category
  Input should be 'billing', 'bug', 'feature_request' or 'other' [type=literal_error, input_value='refund', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error
summary
  Field required [type=missing, input_value={'category': 'refund', 'urgency': 'high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
needs_human
  Field required [type=missing, input_value={'category': 'refund', 'urgency': 'high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing
order_id
  Field required [type=missing, input_value={'category': 'refund', 'urgency': 'high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


# Analysis
Both providers agreed on every field (billing, high, needs_human=True, order ID A-10492) and they differed only in the wording of the summary. Both summaries met the “under 25 words” instruction too. The usage numbers show more differences, and they go straight into the models.py design.

1. Groq used about 4x the tokens for the same task

Gemini: 47 prompt and 79 output tokens, 126 total.
Groq: 307 prompt and 240 completion tokens, 547 total.
The Groq prompt is much bigger, probably because the schema and format instructions count as input.
Of Groq’s 240 completion tokens, 187 were reasoning tokens. That’s about 78% of the output spent thinking about a task that needed one short JSON object. The visible answer was only about 53 tokens.

2. That confirms the rate-limit warning from earlier. At roughly 547 tokens a request, Groq’s 8,000 tokens-per-minute free cap allows about 14 requests a minute, even though the request cap is 30. Gemini’s free tier allows 250,000 input tokens a minute, so requests per minute is the limit there. The retry and concurrency logic will need to track tokens as well as requests on Groq.

# Test Groq with lower reasoning to compare token usage

In [9]:
t0 = time.perf_counter()
q_resp = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "ticket", "strict": True, "schema": strict_schema(Ticket)},
    },
    extra_body={"reasoning_effort": "low"},
)
print(f"{time.perf_counter() - t0:.2f}s")
raw = q_resp.choices[0].message.content
print(raw)
print(Ticket.model_validate_json(raw))   # you validate it yourself
print(q_resp.usage)

0.85s
{
  "category": "billing",
  "urgency": "high",
  "needs_human": true,
  "order_id": "A-10492",
  "summary": "Charged twice for order A-10492, causing overdraft; request immediate refund."
}
category='billing' urgency='high' summary='Charged twice for order A-10492, causing overdraft; request immediate refund.' needs_human=True order_id='A-10492'
CompletionUsage(completion_tokens=108, prompt_tokens=259, total_tokens=367, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=37, rejected_prediction_tokens=None, text_tokens=None), prompt_tokens_details=None, queue_time=0.087988874, prompt_time=0.090894806, completion_time=0.228815806, total_time=0.319710612)


Reasoning tokens dropped about 80%, and total tokens about 33%. At 8,000 tokens a minute, that moves it from roughly 14 requests a minute to roughly 21 on this task. It’s a sample of one, so treat the exact latency as noisy.

A few smaller things in the output:

Visible output went up, from about 53 tokens to about 71 (108 − 37). This time the model pretty-printed the JSON with indentation, and the keys came back in a different order. So never compare raw output strings between runs, only validated objects.
Input tokens fell from 307 to 259 and I can’t tell you why. It may be how Groq counts when the reasoning setting changes. Either way, don’t treat input counts as fixed per prompt.
A caveat on quality: extraction is an easy task, so low effort is likely fine. Harder tasks might suffer, so don’t make low the default without checking on a few harder examples first.

What this means for the gateway. Reasoning effort should be a field on your request model, mapped by each adapter: